# Cache Transfer Workflow

This notebook demonstrates a common collaborative workflow using `CacheStack` and `transfer()`:

1. A **shared global cache** holds approved, vetted results (read-only for regular users).
2. Each user runs under a **local cache** stacked on top of the global one.
   - Cache hits in the global cache are served transparently.
   - New computations land only in the local cache.
3. After reviewing their local results, the user selects which ones to **promote** to the global cache via `transfer()`.

## 1. Setup

We create two caches:

- `_global_cache_rw` — a persistent SQL + PickleFile cache simulating a shared remote store (admin access only)
- `global_cache` — a read-only wrapper that regular users see (writes rejected), created via `.readonly()`
- `local_cache` — a fast in-memory cache for the user's current session
- `user_cache` — local stacked on top of global via `.push()`, so local results take priority

In [1]:
import tempfile
import os
import time

from fleche import fleche, cache
from fleche.caches import Cache
from fleche.storage import Memory
from fleche.storage.pickle_file import PickleFile
from fleche.storage.sql import Sql

No config file found. Using default memory cache.


The admin-accessible cache uses persistent SQL + PickleFile storage.

In [2]:
tmp_dir = tempfile.TemporaryDirectory()

_global_cache_rw = Cache(
    values=PickleFile.with_cloudpickle(tmp_dir.name),
    _calls=Sql(f"sqlite:///{os.path.join(tmp_dir.name, 'global.db')}"),
)
_global_cache_rw

Cache(values=DestructuringStorage(storage=PickleFile(root=PosixPath('/tmp/tmp_hymygkc'), lock_timeout=1.0, lock_wait_start=0.001, secret_key=[], compress=False)), calls=Sql(url='sqlite:////tmp/tmp_hymygkc/global.db', echo=False))

Regular users get a read-only view via `.readonly()`, and a per-session in-memory cache stacked on top via `.push()`.

In [3]:
global_cache = _global_cache_rw.readonly()
local_cache = Cache(values=Memory({}), _calls=Memory({}))
user_cache = global_cache.push(local_cache)
user_cache

CacheStack(stack=(Cache(values=DestructuringStorage(storage=Memory(storage={})), calls=CallStorageAdapter(storage=Memory(storage={}))), ReadOnlyCache(cache=Cache(values=DestructuringStorage(storage=PickleFile(root=PosixPath('/tmp/tmp_hymygkc'), lock_timeout=1.0, lock_wait_start=0.001, secret_key=[], compress=False)), calls=Sql(url='sqlite:////tmp/tmp_hymygkc/global.db', echo=False)))))

## 2. Define Functions

Two `@fleche`-decorated functions simulate a heavy computation and a post-processing step.

In [4]:
@fleche
def simulate(param: float) -> dict:
    """Expensive simulation — takes time, results worth sharing."""
    print(f"  [simulate] Running simulation for param={param}...")
    time.sleep(0.05)  # pretend this is expensive
    return {"param": param, "result": param ** 2 + 1.0}


@fleche
def postprocess(data: dict) -> float:
    """Quick post-processing — user-specific, not worth sharing."""
    print(f"  [postprocess] Processing {data}...")
    return data["result"] * 2.0

() ()
() ()


## 3. Seed the Global Cache (Admin Step)

An admin pre-populates the global cache with approved baseline results.
Regular users never do this — they only read from `global_cache` (the read-only view).

In [5]:
with cache(_global_cache_rw):
    for p in [1.0, 2.0, 3.0]:
        simulate(p)

  [simulate] Running simulation for param=1.0...
  [simulate] Running simulation for param=2.0...
  [simulate] Running simulation for param=3.0...


In [6]:
_global_cache_rw.table()

,name,module,timestart,timestop,walltime
55c426f0f3a90df4dc2416511caf1479221753362100eb143261ba28c98e5125,simulate,__main__,1.774209e+09,1.774209e+09,0.050127
af0fdbd1f495ffdd36b0921657f80d9fe6f78d662877256b6101d02b2d5703d3,simulate,__main__,1.774209e+09,1.774209e+09,0.050137
fef51eb9e42e5b192ed1ace6f10e5396fb544e880cec69fd2622324666735583,simulate,__main__,1.774209e+09,1.774209e+09,0.050157


## 4. User Session

The user runs under `user_cache`, a `CacheStack` with local priority over global.

- Params **already in the global cache** (1.0, 2.0, 3.0) → cache hits, no recomputation.
- **New params** (4.0, 5.0, 6.0) → computed and saved to `local_cache` only.
- Post-processing results also land in `local_cache`.

Params 1–3 are already in the global cache — no `[simulate]` output means they are cache hits.

In [7]:
with cache(user_cache):
    for p in [1.0, 2.0, 3.0]:
        postprocess(simulate(p))

  [postprocess] Processing {'param': 1.0, 'result': 2.0}...


  [postprocess] Processing {'param': 2.0, 'result': 5.0}...
  [postprocess] Processing {'param': 3.0, 'result': 10.0}...


Params 4–6 are new — they will be computed and stored in `local_cache` only.

In [8]:
with cache(user_cache):
    for p in [4.0, 5.0, 6.0]:
        postprocess(simulate(p))

  [simulate] Running simulation for param=4.0...
  [postprocess] Processing {'param': 4.0, 'result': 17.0}...
  [simulate] Running simulation for param=5.0...
  [postprocess] Processing {'param': 5.0, 'result': 26.0}...
  [simulate] Running simulation for param=6.0...
  [postprocess] Processing {'param': 6.0, 'result': 37.0}...


## 5. Inspect the Caches

After the session:
- The **global cache** is unchanged (still only has params 1–3).
- The **local cache** has all new results (simulate + postprocess for params 4–6, and postprocess for 1–3 which were hits from global).

In [9]:
_global_cache_rw.table()

,name,module,timestart,timestop,walltime
55c426f0f3a90df4dc2416511caf1479221753362100eb143261ba28c98e5125,simulate,__main__,1.774209e+09,1.774209e+09,0.050127
af0fdbd1f495ffdd36b0921657f80d9fe6f78d662877256b6101d02b2d5703d3,simulate,__main__,1.774209e+09,1.774209e+09,0.050137
fef51eb9e42e5b192ed1ace6f10e5396fb544e880cec69fd2622324666735583,simulate,__main__,1.774209e+09,1.774209e+09,0.050157


In [10]:
local_cache.table()

,name,module,timestart,timestop,walltime
fef51eb9e42e5b192ed1ace6f10e5396fb544e880cec69fd2622324666735583,simulate,__main__,1.774209e+09,1.774209e+09,0.050157
a7e7621547c87ffe5b44113c4304a72e02933663e089c05c90ca0d381ac40b7c,postprocess,__main__,1.774209e+09,1.774209e+09,0.000096
af0fdbd1f495ffdd36b0921657f80d9fe6f78d662877256b6101d02b2d5703d3,simulate,__main__,1.774209e+09,1.774209e+09,0.050137
5c60bf8e4c8c38e776085e60f886dac7cb84fccc36a54e478d78b50b230c2eee,postprocess,__main__,1.774209e+09,1.774209e+09,0.000085
55c426f0f3a90df4dc2416511caf1479221753362100eb143261ba28c98e5125,simulate,__main__,1.774209e+09,1.774209e+09,0.050127
127cf55fb57db260156dcee43f9e878187a04690e1a7ff1bd9cdbf1b6e401bf9,postprocess,__main__,1.774209e+09,1.774209e+09,0.000033
a1d5a0c10176ed86b738f7ef3be683ffc7a463c573dfad9bc6b75cd9428655c0,simulate,__main__,1.774209e+09,1.774209e+09,0.050143
d75f4f348f079a28de278531e21032f7373b97bfa37ced1ab3ef2fedc120e4c2,postprocess,__main__,1.774209e+09,1.774209e+09,0.000034
b8106539d9a03fb7f63e0429b3fa4ba4f50d8281efc21c534b5f8aa7967c4f87,simulate,__main__,1.774209e+09,1.774209e+09,0.050116
3c2c0d5372cef6e26a39ce8320fe57792e830cc30ed46c1094b927485d47acf2,postprocess,__main__,1.774209e+09,1.774209e+09,0.000043


## 6. Filter Before Transfer

The user reviews their local cache and decides only the `simulate` results are worth promoting to the global cache.
Post-processing results are user-specific and stay local.

`filter()` returns a `FilteredCache` — a read-only view that `transfer()` will iterate over.

In [11]:
simulations_only = local_cache.filter(simulate.fleche.call(param=None))
simulations_only.table()

,name,module,timestart,timestop,walltime
fef51eb9e42e5b192ed1ace6f10e5396fb544e880cec69fd2622324666735583,simulate,__main__,1.774209e+09,1.774209e+09,0.050157
af0fdbd1f495ffdd36b0921657f80d9fe6f78d662877256b6101d02b2d5703d3,simulate,__main__,1.774209e+09,1.774209e+09,0.050137
55c426f0f3a90df4dc2416511caf1479221753362100eb143261ba28c98e5125,simulate,__main__,1.774209e+09,1.774209e+09,0.050127
a1d5a0c10176ed86b738f7ef3be683ffc7a463c573dfad9bc6b75cd9428655c0,simulate,__main__,1.774209e+09,1.774209e+09,0.050143
b8106539d9a03fb7f63e0429b3fa4ba4f50d8281efc21c534b5f8aa7967c4f87,simulate,__main__,1.774209e+09,1.774209e+09,0.050116
1e5f2c348405f55860315db4f04f378b950c8b1b82b8e55e987e602a70e10406,simulate,__main__,1.774209e+09,1.774209e+09,0.050122


## 7. Transfer to the Global Cache

The user (or admin) calls `transfer()` to promote the selected results into the global cache.
With the default `overwrite=False`, entries already present in the global cache are left untouched.
After this, any user will get cache hits for `simulate(4.0)`, `simulate(5.0)`, and `simulate(6.0)` without recomputation.

In [12]:
simulations_only.transfer(_global_cache_rw)

In [13]:
_global_cache_rw.table()

,name,module,timestart,timestop,walltime
1e5f2c348405f55860315db4f04f378b950c8b1b82b8e55e987e602a70e10406,simulate,__main__,1.774209e+09,1.774209e+09,0.050122
55c426f0f3a90df4dc2416511caf1479221753362100eb143261ba28c98e5125,simulate,__main__,1.774209e+09,1.774209e+09,0.050127
a1d5a0c10176ed86b738f7ef3be683ffc7a463c573dfad9bc6b75cd9428655c0,simulate,__main__,1.774209e+09,1.774209e+09,0.050143
af0fdbd1f495ffdd36b0921657f80d9fe6f78d662877256b6101d02b2d5703d3,simulate,__main__,1.774209e+09,1.774209e+09,0.050137
b8106539d9a03fb7f63e0429b3fa4ba4f50d8281efc21c534b5f8aa7967c4f87,simulate,__main__,1.774209e+09,1.774209e+09,0.050116
fef51eb9e42e5b192ed1ace6f10e5396fb544e880cec69fd2622324666735583,simulate,__main__,1.774209e+09,1.774209e+09,0.050157


## 8. Verify: Second User Gets Cache Hits

A second user starts a fresh session with their own empty local cache.
They can now load `simulate(4.0)` – `simulate(6.0)` without any computation.

In [14]:
local_cache_2 = Cache(values=Memory({}), _calls=Memory({}))
user_cache_2 = global_cache.push(local_cache_2)

All six params should now be cache hits — no `[simulate]` output expected.

In [15]:
with cache(user_cache_2):
    for p in [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]:
        simulate(p)

In [16]:
tmp_dir.cleanup()